# Módulo 5 — Conexão com Snowflake

Neste notebook vamos entender a classe de conexão usada no projeto de qualidade de dados
da Equatorial e aprender a usá-la para executar queries e carregar dados em DataFrames.

## O que vamos ver

1. Estrutura do módulo `snowflake_db.py`
2. Autenticação por chave privada (PKI) — padrão corporativo
3. Configuração via variáveis de ambiente (`.env`)
4. Como usar a classe `SnowflakeDB` com context manager
5. Funções utilitárias: `read_sql_to_dataframe` e `write_dataframe_to_snowflake`

---

> **Credenciais Seguras:** 
> - A sua chave privada deve estar em `.secrets/snowflake.p8` na raiz do projeto.
> - As configurações de conta, usuário e senha da chave devem estar no arquivo `.env` na raiz.
>
> **Segurança:** O arquivo `.env` e a pasta `.secrets/` já estão no `.gitignore` para sua proteção.

## 1. Verificando dependências

In [4]:
import importlib

dependencias = {
    "snowflake.connector": "snowflake-connector-python",
    "cryptography": "cryptography",
    "jinja2": "jinja2",
    "pandas": "pandas",
}

todas_ok = True
for modulo, pacote in dependencias.items():
    try:
        importlib.import_module(modulo)
        print(f"  OK  {modulo}")
    except ImportError:
        print(f"  ERRO {modulo} — instale com: uv add {pacote}")
        todas_ok = False

print()
print("Tudo pronto!" if todas_ok else "Instale as dependências acima antes de continuar.")

  OK  snowflake.connector
  OK  cryptography
  OK  jinja2
  OK  pandas

Tudo pronto!


## 2. Importando o módulo do projeto

In [5]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Localizar snowflake_db.py independente do CWD do Jupyter ---
cwd = Path().resolve()
print(f"CWD detectado: {cwd}")

candidatos = [
    cwd,                                              # CWD já é modulo5/
    cwd / "modulo5_snowflake_qualidade",              # CWD é raiz do projeto
    cwd.parent / "modulo5_snowflake_qualidade",       # CWD é outro módulo
    Path(__file__).parent if "__file__" in dir() else cwd,  # via __file__
]

modulo5_dir = next(
    (p for p in candidatos if (p / "snowflake_db.py").exists()),
    None,
)

if modulo5_dir is None:
    raise FileNotFoundError(
        f"snowflake_db.py não encontrado.\n"
        f"CWD={cwd}\n"
        f"Caminhos testados:\n" + "\n".join(f"  {p}" for p in candidatos)
    )

print(f"snowflake_db.py encontrado em: {modulo5_dir}")

# --- Adicionar ao sys.path ---
if str(modulo5_dir) not in sys.path:
    sys.path.insert(0, str(modulo5_dir))

# --- Carregar .env (procura na raiz do projeto) ---
for env_candidate in [cwd / ".env", cwd.parent / ".env", modulo5_dir.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(dotenv_path=env_candidate, override=True)
        print(f".env carregado: {env_candidate}")
        break

# --- Importar o módulo ---
from snowflake_db import (
    SnowflakeDB,
    SNOWFLAKE_CONFIG,
    read_sql_file,
    read_sql_to_dataframe,
    write_dataframe_to_snowflake,
)

import pandas as pd

print(f"\nMódulo importado com sucesso.")
print(f"Conta  : {SNOWFLAKE_CONFIG['account']}")
print(f"Usuário: {SNOWFLAKE_CONFIG['user']}")

CWD detectado: /home/vinicius/projects/t2t/python-learning
snowflake_db.py encontrado em: /home/vinicius/projects/t2t/python-learning/modulo5_snowflake_qualidade
.env carregado: /home/vinicius/projects/t2t/python-learning/.env

Módulo importado com sucesso.
Conta  : khb56279.us-east-1
Usuário: STECH2THINK_SNOWFLAKE@GRUPOEQUATORIALENERGIA.ONMICROSOFT.COM


## 3. Entendendo a autenticação por chave privada

A Equatorial usa **PKI (Public Key Infrastructure)** ao invés de senha para autenticar no Snowflake.
Isso é mais seguro porque:

- A chave privada nunca trafega pela rede
- Cada usuário/serviço tem sua própria chave
- É possível revogar acessos sem mudar senhas

```
Arquivo: rsa_key.p8  (chave privada — NUNCA compartilhe)
   └── Protegida por passphrase
   └── O módulo lê, descriptografa e converte para DER/PKCS8
   └── O conector Snowflake usa os bytes resultantes
```

O fluxo interno da classe `SnowflakeDB`:

In [6]:
# Inspecionando a estrutura da classe sem executar a conexão
import inspect

# Listar os métodos públicos da classe
metodos = [
    (nome, m)
    for nome, m in inspect.getmembers(SnowflakeDB, predicate=inspect.isfunction)
    if not nome.startswith("_")
]

print("Métodos públicos de SnowflakeDB:\n")
for nome, metodo in metodos:
    sig = inspect.signature(metodo)
    doc = (metodo.__doc__ or "").strip().split("\n")[0]
    print(f"  .{nome}{sig}")
    print(f"     → {doc}")
    print()

Métodos públicos de SnowflakeDB:

  .execute_query(self, *args, **kwargs)
     → 

  .query_to_dataframe(self, *args, **kwargs)
     → 

  .write_dataframe(self, *args, **kwargs)
     → 



## 4. Conectando e executando uma query

O padrão recomendado é o **context manager** (`with`).  
Ele garante que a conexão seja fechada mesmo se ocorrer um erro.

```python
with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
    df = db.query_to_dataframe("SELECT ...")
# conexão já está fechada aqui
```

In [7]:
# Teste de conexão — verifique o contexto da sessão
try:
    with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
        df_ctx = db.query_to_dataframe(
            """
            SELECT
                CURRENT_USER()     AS usuario,
                CURRENT_ROLE()     AS role,
                CURRENT_DATABASE() AS database,
                CURRENT_SCHEMA()   AS schema,
                CURRENT_WAREHOUSE() AS warehouse,
                CURRENT_VERSION()  AS versao_snowflake
            """
        )

    print("Conexão bem-sucedida!\n")
    print(df_ctx.T.to_string(header=False))  # exibe na vertical

except Exception as e:
    print(f"Erro: {type(e).__name__}: {e}")
    print("\nVerifique:")
    print("  1. Se o arquivo snowflake.p8 está em .secrets/snowflake.p8")
    print("  2. Se a passphrase e o usuário no arquivo .env estão corretos")
    print("  3. Sua conexão com a internet")

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...
INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
INFO: Connecting to GLOBAL Snowflake domain
INFO: Conexão estabelecida com sucesso.
INFO: Conexão encerrada.


Conexão bem-sucedida!

USUARIO           STECH2THINK_SNOWFLAKE@GRUPOEQUATORIALENERGIA.ONMICROSOFT.COM
ROLE                                                                    PUBLIC
DATABASE                                                          EQTLINFO_HML
SCHEMA                                                                 EQTL_MA
WAREHOUSE                                                          WH_EQTLINFO
VERSAO_SNOWFLAKE                                                        10.9.2


## 5. Usando `read_sql_to_dataframe`

Para análises do dia a dia, use a função utilitária — ela abre e fecha a conexão automaticamente.

In [9]:
# Query direta como string
df_classes = read_sql_to_dataframe(
    """
    SELECT
        TAB_CADASTRO.INSTALACAO AS CR_NUMERO,
        CONSUMIDOR.INSTALACAO_ID AS INSTALACAO_ID,
        TRY_CAST(TAB_CADASTRO.CONTRATO AS INTEGER) AS CONTRATO,
        TRY_CAST(TAB_CADASTRO.CONTA_CONTRATO AS INTEGER) AS CONTA_CONTRATO,
        TRY_CAST(TAB_CADASTRO.MEDIDOR AS INTEGER) AS MEDIDOR_ID,
        TAB_CADASTRO.TIPO_LOCAL_CONSUMO,
        TAB_CADASTRO.GENERO,
        TAB_CADASTRO.DATA_NASCIMENTO,
        TAB_CADASTRO.ESTADO_CIVIL,
        TAB_CADASTRO.LATITUDE,
        TAB_CADASTRO.LONGITUDE,
        TAB_CADASTRO.DATA_INSTALACAO_MEDIDOR
    FROM (
            SELECT *
            FROM EQTLINFO_PRD.EQTL_PA.TAB_CADASTRO
            ORDER BY RANDOM()
            LIMIT 10
    ) TAB_CADASTRO
    LEFT JOIN EQTLINFO_RAW.OPER_PA.CONSUMIDOR CONSUMIDOR 
    ON TAB_CADASTRO.INSTALACAO = LPAD(
            REGEXP_REPLACE(TO_VARCHAR(CONSUMIDOR.CR_NUMERO), '[^0-9]', ''),
            LENGTH(TAB_CADASTRO.INSTALACAO),
            '0'
        )
    """
)

print(f"Linhas retornadas: {len(df_classes)}")
df_classes

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...
INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
INFO: Connecting to GLOBAL Snowflake domain
INFO: Conexão estabelecida com sucesso.
INFO: Conexão encerrada.


Linhas retornadas: 10


,CR_NUMERO,INSTALACAO_ID,CONTRATO,CONTA_CONTRATO,MEDIDOR_ID,TIPO_LOCAL_CONSUMO,GENERO,DATA_NASCIMENTO,ESTADO_CIVIL,LATITUDE,LONGITUDE,DATA_INSTALACAO_MEDIDOR
0,0002997827,None,2997827.0,2.997827e+06,1.304187e+09,Casa,Feminino,1967-03-07,Solteiro(a),-1.3465397,-48.4693652,2015-12-02
1,0012514654,None,503999190.0,3.016030e+09,1.285496e+06,Casa,Masculino,1989-11-02,Outros,-4.253446595,-49.95245604,2015-01-16
2,2001192287,2164342.00,505580892.0,3.028448e+09,2.405130e+10,Casa,Feminino,1969-04-25,Solteiro(a),-1.637585783333,-47.740116983333,2025-12-04
3,0107744126,None,502947758.0,3.007193e+09,1.304051e+09,Casa,Masculino,1956-02-17,Solteiro(a),-1.6073382,-47.4858652,2015-01-29
4,0107194878,None,107194878.0,1.071949e+08,1.101261e+10,Casa,Masculino,1983-08-14,Solteiro(a),-1.4730688,-48.4634136,2020-08-19
5,0080161794,None,NaN,NaN,4.622915e+07,Casa,None,None,None,-2.448236151667,-54.754610240000,1999-04-16
6,2000846667,3098926.00,504380693.0,3.019149e+09,2.100081e+10,Casa,Indefinido,1995-09-18,None,-1.36912788,-48.47156024,2021-09-15
7,2000631786,3098986.00,503703974.0,3.013681e+09,2.100063e+10,Casa,Feminino,1992-02-16,Solteiro(a),-1.711065283852,-48.873500905323,2020-10-27
8,2001257950,3202759.00,505836459.0,3.030408e+09,1.406065e+10,Casa,Masculino,1979-05-13,None,-1.803440090000,-50.721089800000,2024-05-23
9,0017159119,None,NaN,NaN,NaN,Indefinido,None,None,None,-1.173584651410,-48.144065738990,None


## 6. Queries a partir de arquivos `.sql` com templates Jinja2

Para queries mais complexas, o ideal é salvá-las em arquivos `.sql` separados.
O módulo suporta **templates Jinja2** para passar parâmetros dinamicamente.

In [ ]:
# Query parametrizada por CR_NUMERO — lendo de arquivo .sql externo
sql_consumidor = project_root / "queries" / "tb_consumidor_por_cr.sql"

# Definir o CR_NUMERO que queremos consultar
cr_numero = "0002997827"

# read_sql_file processa o template Jinja2 e substitui {{ cr_numero }}
query_renderizada = read_sql_file(sql_consumidor, cr_numero=cr_numero)

print(f"Buscando consumidor: CR_NUMERO = {cr_numero}")
print(f"Arquivo SQL         : {sql_consumidor.name}")
print()
print("Query renderizada:")
print(query_renderizada)

In [ ]:
# Executar a query e trazer o cadastro completo do consumidor
df_consumidor = read_sql_to_dataframe(sql_consumidor, cr_numero=cr_numero)

print(f"Registros encontrados: {len(df_consumidor)}")
df_consumidor.T  # exibe na vertical para facilitar a leitura

## 7. Queries parametrizadas (seguras contra SQL injection)

Para valores dinâmicos dentro da query, use o mecanismo de parâmetros `%s` do conector
ao invés de f-strings. Isso previne **SQL injection**.

In [ ]:
# Comparação: ERRADO vs CORRETO

cod_uf = "MA"
limite = 20

# ❌ ERRADO — vulnerável a SQL injection
# query_errada = f"SELECT * FROM CONSUMIDORES_UC WHERE COD_UF = '{cod_uf}' LIMIT {limite}"

# ✅ CORRETO — valores passados como parâmetros separados
query_segura = """
    SELECT
        COD_CONSUMIDOR,
        NOM_CONSUMIDOR,
        COD_CLASSE_CONSUMO,
        VLR_CONSUMO_MEDIO_KWH
    FROM CONSUMIDORES_UC
    WHERE COD_UF = %s
      AND FLG_ATIVO = TRUE
    ORDER BY VLR_CONSUMO_MEDIO_KWH DESC
    LIMIT %s
"""

try:
    with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
        colunas, dados = db.execute_query.__wrapped__(db, query_segura)  # sem param direto
        # Para queries parametrizadas, use o cursor diretamente:
        db.cursor.execute(query_segura, (cod_uf, limite))
        colunas = [desc[0] for desc in db.cursor.description]
        dados = db.cursor.fetchall()

    df_param = pd.DataFrame(dados, columns=colunas)
    print(f"Top {limite} consumidores de {cod_uf} por consumo médio:")
    print(df_param)

except Exception as e:
    print(f"Erro: {e}")

## 8. Gravando resultados de volta ao Snowflake

In [ ]:
import pandas as pd
from datetime import datetime

# Exemplo: criar um DataFrame com resultado de análise de qualidade
# e gravar na tabela de resultados

df_qualidade = pd.DataFrame({
    "COD_UF": ["MA", "PA", "GO"],
    "QTD_UCS": [150000, 120000, 95000],
    "PCT_SEM_CPF": [4.2, 5.8, 2.1],
    "PCT_SEM_MEDIDOR": [1.3, 2.0, 0.8],
    "SCORE_QUALIDADE": [87.5, 82.3, 93.1],
    "DAT_PROCESSAMENTO": datetime.now(),
})

print("DataFrame a ser gravado:")
print(df_qualidade)

# Para gravar de fato, descomente:
# write_dataframe_to_snowflake(
#     df_qualidade,
#     table_name="TB_RESULTADO_QUALIDADE_CADASTRO",
#     auto_create_table=True,
#     overwrite=False,
# )
# print("Dados gravados com sucesso!")

## 9. Boas práticas — resumo

| Prática | Por quê |
|---------|--------|
| Usar `with SnowflakeDB(...) as db:` | Fecha a conexão automaticamente, mesmo com erro |
| Parâmetros via `%s`, não f-string | Previne SQL injection e problemas com caracteres especiais |
| Queries em arquivos `.sql` | Facilita versionamento, revisão e reuso |
| Templates Jinja2 para listas | Gera cláusulas `IN (...)` corretamente sem concatenação manual |
| Nunca commitar `rsa_key.p8` | Chave privada comprometida = acesso ao banco comprometido |
| `LIMIT` em exploração | Snowflake cobra por dados scaneados — limite durante desenvolvimento |
| `write_pandas` com `chunk_size` | Evita timeout e erros de memória em cargas grandes |